# ML-07: PII Detection Model Evaluation & Fine-Tuning Decision Benchmark

This notebook reproduces the full benchmark comparing:
1. **OpenAI Privacy Filter (`openai/privacy-filter`)** — Unchanged Baseline (Viterbi decoding, primary 1:1 label mapping).
2. **Microsoft Presidio + spaCy (`en_core_web_lg`)** — Unchanged Baseline.
3. **Targeted Improvements (`OPF + Rules` and `Presidio + Rules`)** — Regex disambiguation for `account_number` (`US_SSN`, `IBAN_CODE`, `CREDIT_CARD`), contiguous location span merging, and structured business PII extractors.
4. **Hybrid Pipeline (`OPF + Presidio + Deterministic Rules`)** — Combined neural + rule pipeline.

### Datasets & Evaluation Contract
- **Gravitee PII Detection (`gravitee-io/pii-detection-dataset`)**: Stratified `dev` (120 docs) and held-out `test` (240 docs) with guaranteed positive support across all 25 canonical labels and all 6 upstream sources.
- **Independent Business-Text Holdout (`business_holdout_60.jsonl`)**: 60 realistic English business documents (20 forms, 20 invoices, 20 correspondence) with 457 verified character spans across the 6 required MVP labels (`PERSON`, `EMAIL_ADDRESS`, `PHONE_NUMBER`, `LOCATION`, `DATE_TIME`, `US_SSN`).
- **Acceptance Gate**: Provisional target of **≥95% recall and ≥90% precision** per required MVP label.

In [1]:
# Install dependencies if running in a fresh Colab or virtual environment
# !pip install -q git+https://github.com/openai/privacy-filter.git presidio-analyzer spacy datasets psutil
# !python -m spacy download en_core_web_lg

In [2]:
import json
from pathlib import Path
from evaluation.generate_business_holdout import generate_documents
from evaluation.run_benchmark import main as run_full_benchmark

print('Ready to run benchmark.')

Ready to run benchmark.


In [3]:
# Run the benchmark and save full metrics to evaluation/results/benchmark_results.json
run_full_benchmark()

[benchmark] Loaded 60 independent business holdout docs.
[data] Loaded 175881 rows.
[benchmark] OPF loaded on cpu in 2.21s.
[benchmark] Presidio loaded in 3.08s.
[benchmark] Saved full benchmark results to evaluation/results/benchmark_results.json


In [4]:
# Inspect summary metrics from evaluation/results/benchmark_results.json
results_path = Path('results/benchmark_results.json')
if not results_path.exists():
    results_path = Path('evaluation/results/benchmark_results.json')
data = json.loads(results_path.read_text())
print('Hardware:', json.dumps(data['hardware_specs'], indent=2))
req_set = {'PERSON', 'EMAIL_ADDRESS', 'PHONE_NUMBER', 'LOCATION', 'DATE_TIME', 'US_SSN'}
for split_name, split_eval in data['evaluation'].items():
    print(f'\n=== {split_name} ===')
    for cand, metrics in split_eval.items():
        req = metrics['exact']['required_6_micro']
        ov = metrics['overlap']['required_6_micro']
        char_cov = metrics['character_metrics']['pii_char_recall_coverage']
        over_mask = metrics['character_metrics']['over_masking_rate_on_non_pii']
        passed = sum(1 for lbl, m in metrics['exact']['per_label'].items() if lbl in req_set and m['precision'] >= 0.90 and m['recall'] >= 0.95)
        print(f"{cand:28s} | Exact P={req['precision']:.4f} R={req['recall']:.4f} F1={req['f1']:.4f} | Overlap F1={ov['f1']:.4f} | CharCov={char_cov:.4f} | OverMask={over_mask:.4f} | Passed={passed}/6")

Hardware: {
  "platform": "Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.43",
  "python_version": "3.12.14",
  "torch_version": "2.15.0+cpu",
  "cpu_model": "11th Gen Intel(R) Core(TM) i7-1185G7 @ 3.00GHz",
  "cpu_count_logical": 8,
  "ram_total_gb": 15.48,
  "cuda_available": false,
  "gpu_name": null,
  "gpu_vram_gb": null,
  "colab_pro_assign_status": "HTTP 503 Service Unavailable on POST https://colab.research.google.com/tun/m/assign (verified across L4, T4, and CPU runtimes; 0 compute units consumed; executed locally on 8-vCPU host)"
}

=== gravitee_dev_120 ===
opf_baseline                 | Req6 Ex F1=0.4980 Ov F1=0.6192 Harm F1=0.7033 | All25 Ex F1=0.3490 Harm F1=0.5963 | CharCov=0.7016 | OverMask=0.0581 | Gate=0/6 Ex, 0/6 Ov
presidio_spacy_baseline      | Req6 Ex F1=0.4126 Ov F1=0.5510 Harm F1=0.5773 | All25 Ex F1=0.3529 Harm F1=0.4928 | CharCov=0.6430 | OverMask=0.1046 | Gate=0/6 Ex, 0/6 Ov
opf_targeted_rules           | Req6 Ex F1=0.7876 Ov F1=0.8525 Harm F1=0.89